In [4]:
import pandas as pd

# Q1
roll_number = "1024170240"

fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.",
     "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.",
     "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.",
     "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.",
     "keywords": "pay payment upi fee", "category": "billing"},
]

categories = ["billing", "account", "general"]
last_two = [int(d) for d in roll_number[-2:]]
for d in last_two:
    print("digit", d, "->", categories[d % 3])

# digit 4 -> account, digit 0 -> billing
my_entries = [
    {"question": "how do i update my registered mobile number",
     "answer": "Go to Profile > Edit Details and enter your new mobile number.",
     "keywords": "mobile number update", "category": categories[last_two[0] % 3]},
    {"question": "how do i get a refund",
     "answer": "Refunds are processed within 7 working days to your original payment method.",
     "keywords": "refund money return", "category": categories[last_two[1] % 3]},
]

df = pd.DataFrame(fixed_entries + my_entries)
df

digit 4 -> account
digit 0 -> billing


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,how do i update my registered mobile number,Go to Profile > Edit Details and enter your ne...,mobile number update,account
5,how do i get a refund,Refunds are processed within 7 working days to...,refund money return,billing


In [5]:
# Q2
def score_query(query, df):
    query_words = set(query.lower().split())
    results = []
    for i, row in df.iterrows():
        keywords = set(row["keywords"].split())
        matches = len(query_words & keywords)
        if matches > 0:
            confidence = round(matches / len(keywords), 2)
            results.append({"question": row["question"], "answer": row["answer"],
                            "matches": matches, "confidence": confidence})
    if not results:
        return pd.DataFrame(columns=["question", "answer", "matches", "confidence"])
    ranked = pd.DataFrame(results).sort_values(by="confidence", ascending=False)
    return ranked.reset_index(drop=True)

score_query("how to pay the fee using upi", df)

,question,answer,matches,confidence
0,how can i pay the fee,"You can pay via UPI, card, or net banking.",3,0.75
1,what is the annual fee,The annual fee is Rs 500.,1,0.25


In [6]:
# Q3
def same_category(category_name, df):
    return df[df["category"] == category_name]["question"]

print(same_category(my_entries[0]["category"], df))

1                          how to reset password
4    how do i update my registered mobile number
Name: question, dtype: str


In [7]:
# Q4
new_keyword = input("Enter a new keyword for entry 4 (mobile number question): ")
df.loc[4, "keywords"] = df.loc[4, "keywords"] + " " + new_keyword.strip().lower()
df.to_csv(roll_number + "_faq_data.csv", index=False)
df

Enter a new keyword for entry 4 (mobile number question):  phone


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,how do i update my registered mobile number,Go to Profile > Edit Details and enter your ne...,mobile number update phone,account
5,how do i get a refund,Refunds are processed within 7 working days to...,refund money return,billing


In [8]:
# Q5
print(df.groupby("category").size())

category
account    2
billing    3
general    1
dtype: int64


In [9]:
# Q6
def best_match(query, df):
    ranked = score_query(query, df)
    print("Query:", query)
    if ranked.empty:
        print("No matching entry found.")
        return
    top_score = ranked["confidence"].max()
    top = ranked[ranked["confidence"] == top_score]
    if len(top) > 1:
        print("Tie between", len(top), "entries with confidence", top_score)
        for i, row in top.iterrows():
            print(" -", row["question"], "->", row["answer"])
    else:
        print("Best match:", top.iloc[0]["question"], "->", top.iloc[0]["answer"],
              "(confidence", str(top_score) + ")")

best_match("what is the fee", df)            # tie
print()
best_match("how to reset my password", df)   # no tie

Query: what is the fee
Tie between 2 entries with confidence 0.25
 - what is the annual fee -> The annual fee is Rs 500.
 - how can i pay the fee -> You can pay via UPI, card, or net banking.

Query: how to reset my password
Best match: how to reset password -> Go to Settings > Reset Password. (confidence 0.67)
